# 01 — Animal Overview Reports

Per-animal PDF reports with session timeline, psychometric curves (pairwise),
update matrices, Hard-A vs Hard-B comparisons, and strip-plot summary.

**Structure:**
1. `compute_phase` — data for one animal × one phase
2. `plot_*` — individual figure per page type
3. `generate_animal_report` — assembles all pages for one animal
4. Batch loop — saves one PDF per animal

## Setup

In [1]:
%matplotlib inline
from shared_setup import *
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D
from collections import OrderedDict

experiment, info = load_data()
print(f"Mode: {info['mode']}")

Loaded snapshot: 22 animals, 1185 sessions (exported 2026-06-03)
Mode: snapshot


/Users/Serkan/Desktop/pro/PhD/main/repos/sound_categorisation/notebooks/shared_setup.py:192: UserWarning: Config has changed since snapshot was exported. Re-export if column mappings changed.
  experiment, meta = load_snapshot(


## Cohort Summary

In [ ]:
rows = []
for aid, animal in sorted(experiment.animals.items()):
    table = animal.session_table
    types = dict(table.session_type.value_counts())
    dists = dict(table.distribution.value_counts())
    rows.append({
        'animal_id': aid, 'genotype': animal.genotype,
        'n_sessions': len(table),
        'n_uniform': dists.get('Uniform', 0),
        'n_hard_a': dists.get('Hard-A', 0),
        'n_hard_b': dists.get('Hard-B', 0),
        'regular': types.get('regular', 0),
        'masking': types.get('masking', 0),
        'opto': types.get('opto', 0),
        'washout': types.get('washout', 0),
    })
cohort_df = pd.DataFrame(rows)
print(cohort_df.to_string(index=False))

## Condition Definitions

In [ ]:
OPTO_UNIFORM = OrderedDict([
    ('baseline',  ('uniform_training_last5', None)),
    ('masking',   ('uniform_masking',        None)),
    ('all_opto',  ('uniform_opto',           'all')),
    ('opto_off',  ('uniform_opto',           None)),
    ('opto_on',   ('uniform_opto',           'opto_on')),
    ('post_opto', ('uniform_opto',           'post_opto')),
])
OPTO_HARD_A = OrderedDict([
    ('masking',   ('hard_a_masking', None)),
    ('all_opto',  ('hard_a_opto',    'all')),
    ('opto_off',  ('hard_a_opto',    None)),
    ('opto_on',   ('hard_a_opto',    'opto_on')),
    ('post_opto', ('hard_a_opto',    'post_opto')),
])
OPTO_HARD_B = OrderedDict([
    ('masking',   ('hard_b_masking', None)),
    ('all_opto',  ('hard_b_opto',    'all')),
    ('opto_off',  ('hard_b_opto',    None)),
    ('opto_on',   ('hard_b_opto',    'opto_on')),
    ('post_opto', ('hard_b_opto',    'post_opto')),
])
REGULAR_UNIFORM = OrderedDict([('baseline', ('uniform_training_last5', None))])
REGULAR_HARD_A  = OrderedDict([('regular',  ('hard_a_regular', None))])
REGULAR_HARD_B  = OrderedDict([('regular',  ('hard_b_regular', None))])

PHASE_CONDITIONS = {
    'opto':     {'uniform': OPTO_UNIFORM, 'hard_a': OPTO_HARD_A, 'hard_b': OPTO_HARD_B},
    'non-opto': {'uniform': REGULAR_UNIFORM, 'hard_a': REGULAR_HARD_A, 'hard_b': REGULAR_HARD_B},
}

CC = {
    'baseline': '#1f77b4', 'regular': '#1f77b4', 'masking': '#ff7f0e',
    'all_opto': '#2ca02c', 'opto_off': '#9467bd', 'opto_on': '#d62728',
    'post_opto': '#8c564b',
}
HARD_A_COLOUR = '#d62728'
HARD_B_COLOUR = '#2ca02c'

MIN_TRIALS = 10
N_BOOTSTRAP = 200

## Compute

In [ ]:
def _resolve_mask_fn(key):
    if key is None:
        return None
    if key == 'all':
        return lambda s: build_mask(s.trials, exclude_opto=False)
    if key == 'opto_on':
        return lambda s: opto_mask(s.trials, delta=0)
    if key == 'post_opto':
        return lambda s: opto_mask(s.trials, delta=1)
    raise ValueError(f"Unknown mask key: {key}")


def is_opto_cohort(animal):
    return animal.session_table['session_type'].isin(['opto', 'masking']).any()


def compute_phase(animal, phase, cohort=None):
    if cohort is None:
        cohort = 'opto' if is_opto_cohort(animal) else 'non-opto'
    conditions = PHASE_CONDITIONS[cohort][phase]
    clean, psyc, um = {}, {}, {}
    for label, (preset, mask_key) in conditions.items():
        sessions = select_sessions(animal, preset)
        if not sessions:
            clean[label] = psyc[label] = um[label] = None
            continue
        filtered = filter_trials(sessions, mask_fn=_resolve_mask_fn(mask_key),
                                 min_trials=MIN_TRIALS)
        if not filtered:
            clean[label] = psyc[label] = um[label] = None
            continue
        clean[label] = filtered
        try:
            psyc[label] = compute_psychometric(filtered, mode='pooled',
                                                n_bins=8, n_bootstrap=N_BOOTSTRAP)
        except Exception:
            psyc[label] = None
        try:
            um[label] = compute_um(filtered)
        except Exception:
            um[label] = None
    return clean, psyc, um

### Downsampling

In [ ]:
from behav_utils.analysis.psychometry import _fit_psychometric_once
from behav_utils.analysis.update_matrix import fit_psychometric
import warnings

def _pool_stim_choice(sessions):
    """Pool valid (stimulus, choice) arrays from filtered sessions."""
    stim, choice = [], []
    for sess in sessions:
        a = sess.get_arrays()
        mask = ~a['no_response']
        stim.append(a['stimuli'][mask])
        choice.append(a['choices'][mask])
    if not stim:
        return None, None
    return np.concatenate(stim), np.concatenate(choice)


def _extract_pairs(sessions, trial_filter='post_correct'):
    """Extract valid consecutive (prev_stim, curr_stim, curr_choice) pairs."""
    ps, cs, cc = [], [], []
    for sess in sessions:
        a = sess.get_arrays()
        stim = a['stimuli']
        choice = a['choices']
        cat = a['categories']
        no_resp = a['no_response']
        n = len(stim)
        if n < 2:
            continue
        rewards = (choice == cat).astype(float)
        rewards[np.isnan(choice)] = np.nan
        for i in range(1, n):
            if no_resp[i] or no_resp[i - 1]:
                continue
            if trial_filter == 'post_correct' and rewards[i - 1] != 1:
                continue
            ps.append(stim[i - 1])
            cs.append(stim[i])
            cc.append(choice[i])
    if not ps:
        return None, None, None
    return np.array(ps), np.array(cs), np.array(cc)


def _um_from_pairs(prev_stim, curr_stim, curr_choice, n_bins=8):
    """Compute UM directly from pre-selected pairs."""
    bin_edges = np.linspace(-1, 1, n_bins + 1)
    midpoints = (bin_edges[:-1] + bin_edges[1:]) / 2
    prev_bins = np.clip(np.digitize(prev_stim, bin_edges) - 1, 0, n_bins - 1)

    total_psych = fit_psychometric(curr_stim, curr_choice, midpoints)
    total_curve = (total_psych['y_fit']
                   if total_psych['success']
                   else np.full(n_bins, np.nan))

    um = np.full((n_bins, n_bins), np.nan)
    for j in range(n_bins):
        mask = prev_bins == j
        if mask.sum() < 10:
            continue
        cond_psych = fit_psychometric(curr_stim[mask], curr_choice[mask], midpoints)
        if cond_psych['success']:
            um[:, j] = cond_psych['y_fit'] - total_curve
    return um


def downsample_phase(clean_dict, n_repeats=100, n_bins=8, seed=42):
    """
    Downsample all conditions to matched n, compute psychometric + UM.

    Parameters
    ----------
    clean_dict : dict[label -> list[SessionData] | None]
        From compute_phase.
    n_repeats : int
        Number of subsample repeats to average.
    n_bins : int
    seed : int

    Returns
    -------
    psyc_dict : dict[label -> psyc result compatible with plot_psychometric]
    um_dict   : dict[label -> um result compatible with plot_um]
    """
    rng = np.random.default_rng(seed)

    # ── Pool trials and pairs per condition ─────────────────────
    trials_pool = {}   # label -> (stim, choice)
    pairs_pool = {}    # label -> (prev_stim, curr_stim, curr_choice)

    for label, sessions in clean_dict.items():
        if sessions is None:
            continue
        s, c = _pool_stim_choice(sessions)
        if s is not None and len(s) > 0:
            trials_pool[label] = (s, c)
        ps, cs, cc = _extract_pairs(sessions)
        if ps is not None and len(ps) > 0:
            pairs_pool[label] = (ps, cs, cc)

    # ── Find min n ──────────────────────────────────────────────
    if not trials_pool:
        return {l: None for l in clean_dict}, {l: None for l in clean_dict}

    min_n_trials = min(len(s) for s, c in trials_pool.values())
    min_n_pairs = (min(len(ps) for ps, cs, cc in pairs_pool.values())
                   if pairs_pool else 0)

    x_fit = np.linspace(-1, 1, 100)
    bin_edges = np.linspace(-1, 1, n_bins + 1)
    bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2

    # ── Subsample psychometric K times ──────────────────────────
    psyc_dict = {}
    for label in clean_dict:
        if label not in trials_pool:
            psyc_dict[label] = None
            continue

        stim_all, choice_all = trials_pool[label]
        all_bin_means = []
        all_y_fit = []
        all_params = []

        for _ in range(n_repeats):
            idx = rng.choice(len(stim_all), min_n_trials, replace=True)
            s_sub, c_sub = stim_all[idx], choice_all[idx]
            fit = _fit_psychometric_once(s_sub, c_sub, x_eval=x_fit)
            if not fit.get('success', False):
                continue
            all_y_fit.append(fit['y_fit'])
            all_params.append({k: fit[k] for k in ('mu', 'sigma', 'lapse_low', 'lapse_high')})

            # Bin means for this subsample
            bi = np.clip(np.digitize(s_sub, bin_edges) - 1, 0, n_bins - 1)
            bm = np.full(n_bins, np.nan)
            for b in range(n_bins):
                m = bi == b
                if m.sum() > 0:
                    bm[b] = np.nanmean(c_sub[m])
            all_bin_means.append(bm)

        if not all_y_fit:
            psyc_dict[label] = None
            continue

        avg_params = {k: float(np.nanmean([p[k] for p in all_params]))
                      for k in ('mu', 'sigma', 'lapse_low', 'lapse_high')}
        y_arr = np.array(all_y_fit)
        bm_arr = np.array(all_bin_means)

        psyc_dict[label] = {
            'mode': 'pooled', 'success': True,
            'params': avg_params,
            'params_ci': {k: (float(np.nanpercentile([p[k] for p in all_params], 2.5)),
                              float(np.nanpercentile([p[k] for p in all_params], 97.5)))
                          for k in avg_params},
            'x_fit': x_fit,
            'y_fit': np.nanmean(y_arr, axis=0),
            'curve_band': {
                'x': x_fit,
                'median': np.nanmedian(y_arr, axis=0),
                'lo': np.nanpercentile(y_arr, 2.5, axis=0),
                'hi': np.nanpercentile(y_arr, 97.5, axis=0),
            },
            'bin_centres': bin_centres,
            'bin_means': np.nanmean(bm_arr, axis=0),
            'bin_counts': np.full(n_bins, min_n_trials // n_bins),
            'n_trials': min_n_trials,
            'n_fits': len(all_y_fit),
        }

    # ── Subsample UM K times ────────────────────────────────────
    um_dict = {}
    for label in clean_dict:
        if label not in pairs_pool or min_n_pairs < 80:
            um_dict[label] = None
            continue

        ps_all, cs_all, cc_all = pairs_pool[label]
        all_um = []

        for _ in range(n_repeats):
            idx = rng.choice(len(ps_all), min_n_pairs, replace=False)
            um = _um_from_pairs(ps_all[idx], cs_all[idx], cc_all[idx], n_bins)
            all_um.append(um)

        if not all_um:
            um_dict[label] = None
            continue

        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            avg_um = np.nanmean(np.stack(all_um), axis=0)

        um_dict[label] = {
            'um': avg_um,
            'n_bins': n_bins,
            'n_trials': min_n_pairs,
            'n_sessions': 0,
            'method': 'downsampled',
            'info': {'n_repeats': n_repeats},
        }

    # Fill missing
    for label in clean_dict:
        if label not in psyc_dict:
            psyc_dict[label] = None
        if label not in um_dict:
            um_dict[label] = None

    return psyc_dict, um_dict


## Plot Functions

In [ ]:
def _plot_pair(ax, psyc_dict, key_a, key_b, title,
               colour_a=None, colour_b=None, label_a=None, label_b=None):
    pa, pb = psyc_dict.get(key_a), psyc_dict.get(key_b)
    ca = colour_a or CC.get(key_a, 'grey')
    cb = colour_b or CC.get(key_b, 'grey')
    la = label_a or key_a
    lb = label_b or key_b
    if pa and pa.get('success'):
        plot_psychometric(pa, ax=ax, color=ca, label=f'{la} (n={pa["n_trials"]})')
    if pb and pb.get('success'):
        plot_psychometric(pb, ax=ax, color=cb, label=f'{lb} (n={pb["n_trials"]})')
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=7)

In [ ]:
def plot_timeline(animal, aid):
    table = animal.session_table
    fig, ax = plt.subplots(figsize=(12, 3))
    dist_c = {'Uniform': '#1f77b4', 'Hard-A': '#d62728', 'Hard-B': '#2ca02c'}
    type_m = {'regular': 'o', 'masking': 's', 'opto': '^', 'washout': 'D'}
    for _, row in table.iterrows():
        ax.scatter(row['session_idx'], row['accuracy'],
                   c=dist_c.get(row['distribution'], 'grey'),
                   marker=type_m.get(row['session_type'], 'o'),
                   s=60, edgecolors='k', linewidths=0.5, zorder=3)
    ax.set_xlabel('Session index'); ax.set_ylabel('Accuracy')
    ax.set_ylim(0.4, 1.0); ax.axhline(0.5, ls='--', c='grey', lw=0.5)
    handles = ([Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=8, label=d)
                for d, c in dist_c.items()] +
               [Line2D([0],[0], marker=m, color='w', markerfacecolor='grey', markersize=8, label=t)
                for t, m in type_m.items()])
    ax.legend(handles=handles, ncol=4, fontsize=8, loc='lower right')
    fig.suptitle(f'{aid} ({animal.genotype.upper()}) — Session Timeline', fontsize=14)
    fig.tight_layout()
    return fig

In [ ]:
def plot_uniform_psychometric(psyc, tag):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    _plot_pair(axes[0,0], psyc, 'baseline', 'masking',   'Baseline vs Masking')
    _plot_pair(axes[0,1], psyc, 'masking',  'all_opto',  'Masking vs All Opto')
    _plot_pair(axes[0,2], psyc, 'masking',  'opto_off',  'Masking vs Opto-Off')
    _plot_pair(axes[1,0], psyc, 'opto_off', 'opto_on',   'Opto-Off vs Opto-On')
    _plot_pair(axes[1,1], psyc, 'opto_on',  'post_opto', 'Opto-On vs Post-Opto')
    _plot_pair(axes[1,2], psyc, 'opto_off', 'post_opto', 'Opto-Off vs Post-Opto')
    fig.suptitle(f'{tag} — Uniform Psychometric', fontsize=14)
    fig.tight_layout()
    return fig

In [ ]:
def plot_uniform_um(um, tag):
    order = ['baseline', 'masking', 'all_opto', 'opto_off', 'opto_on', 'post_opto']
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for i, key in enumerate(order):
        ax = axes[i // 3, i % 3]
        u = um.get(key)
        if u is not None:
            plot_um(u, ax=ax, title=f'{key} (n={u["n_trials"]})',
                    vmin=-0.35, vmax=0.35, colorbar=False)
        else:
            ax.text(0.5, 0.5, f'{key}\nNo data', ha='center', va='center',
                    transform=ax.transAxes, color='grey')
            ax.set_axis_off()
    fig.suptitle(f'{tag} — Uniform Update Matrices', fontsize=14)
    fig.tight_layout()
    return fig

In [ ]:
def plot_hard_psychometric(psyc_a, psyc_b, tag):
    fig, axes = plt.subplots(2, 4, figsize=(22, 10))
    _plot_pair(axes[0,0], psyc_a, 'masking', 'opto_off', 'Hard-A: Masking vs Opto-Off')
    _plot_pair(axes[0,1], psyc_b, 'masking', 'opto_off', 'Hard-B: Masking vs Opto-Off')
    for col, key, title in [(2, 'masking', 'A vs B: Masking'),
                             (3, 'opto_off', 'A vs B: Opto-Off')]:
        ax = axes[0, col]
        pa, pb = psyc_a.get(key), psyc_b.get(key)
        if pa and pa.get('success'):
            plot_psychometric(pa, ax=ax, color=HARD_A_COLOUR, label=f'Hard-A (n={pa["n_trials"]})')
        if pb and pb.get('success'):
            plot_psychometric(pb, ax=ax, color=HARD_B_COLOUR, label=f'Hard-B (n={pb["n_trials"]})')
        ax.set_title(title, fontsize=11); ax.legend(fontsize=7)
    _plot_pair(axes[1,0], psyc_a, 'opto_off', 'opto_on',   'Hard-A: Opto-Off vs Opto-On')
    _plot_pair(axes[1,1], psyc_b, 'opto_off', 'opto_on',   'Hard-B: Opto-Off vs Opto-On')
    _plot_pair(axes[1,2], psyc_a, 'opto_on',  'post_opto', 'Hard-A: Opto-On vs Post-Opto')
    _plot_pair(axes[1,3], psyc_b, 'opto_on',  'post_opto', 'Hard-B: Opto-On vs Post-Opto')
    fig.suptitle(f'{tag} — Hard Psychometric', fontsize=14)
    fig.tight_layout()
    return fig

In [ ]:
def plot_hard_um(um_a, um_b, tag):
    order = ['masking', 'all_opto', 'opto_off', 'opto_on', 'post_opto']
    fig, axes = plt.subplots(2, 5, figsize=(24, 10))
    for col, key in enumerate(order):
        for row, (um, dist_label) in enumerate([(um_a, 'Hard-A'), (um_b, 'Hard-B')]):
            ax = axes[row, col]
            u = um.get(key)
            if u is not None:
                plot_um(u, ax=ax, title=f'{dist_label} {key} (n={u["n_trials"]})',
                        vmin=-0.35, vmax=0.35, colorbar=False)
            else:
                ax.text(0.5, 0.5, f'{dist_label} {key}\nNo data', ha='center',
                        va='center', transform=ax.transAxes, color='grey')
                ax.set_axis_off()
    fig.suptitle(f'{tag} — Hard Update Matrices', fontsize=14)
    fig.tight_layout()
    return fig

In [ ]:
PHASE_ORDER = ['uniform', 'hard_a', 'hard_b']
PHASE_LABELS = {'uniform': 'Uniform', 'hard_a': 'Hard-A', 'hard_b': 'Hard-B'}
PHASE_GAP = 1.5
BOX_WIDTH = 0.6
DOT_SIZE = 25
DOT_ALPHA = 0.7


def build_strip_df(animal, phases=PHASE_ORDER,
                   stats_of_interest=('accuracy', 'win_stay', 'lose_shift'),
                   cohort=None):
    """
    Per-session DataFrame for strip plots.
    Each row = one session in one condition.
    """
    rows = []
    for phase in phases:
        clean, _, _ = compute_phase(animal, phase, cohort)
        for condition, sessions in clean.items():
            if sessions is None:
                continue
            for sess in sessions:
                n_trials = int(sess.trials.valid_mask.sum())
                # n_trials is computed directly, not a registered stat
                registered = [s for s in stats_of_interest if s != 'n_trials']
                features = compute_session_features(sess, stat_names=registered) if registered else {}
                row = {'phase': phase, 'condition': condition, 'n_trials': n_trials}
                for stat in stats_of_interest:
                    if stat != 'n_trials':
                        row[stat] = features.get(stat, np.nan)
                rows.append(row)
    return pd.DataFrame(rows)


def plot_strip(stats_df, stats_to_plot, title, phase_order=PHASE_ORDER):
    """
    Grouped box + dots. Returns list[Figure], one page per stat.

    Parameters
    ----------
    stats_df : DataFrame with 'phase', 'condition', and stat columns.
    stats_to_plot : list of (column_name, display_label).
    title : str
    """
    rng = np.random.default_rng(42)
    figures = []

    seen = []
    for _, row in stats_df.iterrows():
        c = row['condition']
        if c not in seen and c in CC:
            seen.append(c)
    legend_handles = [Line2D([0],[0], marker='o', color='w',
                             markerfacecolor=CC[c], markersize=8, label=c)
                      for c in seen]

    for col, ylabel in stats_to_plot:
        fig, ax = plt.subplots(figsize=(10, 5))

        if col not in stats_df.columns or stats_df[col].dropna().empty:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                    transform=ax.transAxes, color='grey', fontsize=14)
            ax.set_ylabel(ylabel)
            fig.suptitle(f'{title} — {ylabel}', fontsize=14)
            figures.append(fig)
            continue

        x_pos = 0
        tick_positions, tick_labels = [], []
        group_spans = []

        for phase in phase_order:
            phase_df = stats_df[stats_df['phase'] == phase]
            if phase_df.empty:
                continue
            conditions = [c for c in phase_df['condition'].unique() if c in CC]
            if not conditions:
                continue

            group_start = x_pos
            for cond in conditions:
                vals = phase_df.loc[phase_df['condition'] == cond, col].dropna().values
                if len(vals) == 0:
                    x_pos += 1
                    continue

                colour = CC[cond]
                bp = ax.boxplot(
                    [vals], positions=[x_pos], widths=BOX_WIDTH,
                    patch_artist=True, showfliers=False,
                    medianprops=dict(color='k', linewidth=1.5),
                    whiskerprops=dict(color=colour),
                    capprops=dict(color=colour),
                )
                bp['boxes'][0].set_facecolor(colour)
                bp['boxes'][0].set_alpha(0.35)
                bp['boxes'][0].set_edgecolor(colour)

                jitter = rng.normal(0, 0.07, len(vals))
                ax.scatter(x_pos + jitter, vals, s=DOT_SIZE, c=colour,
                           edgecolors='white', linewidths=0.4,
                           alpha=DOT_ALPHA, zorder=3)

                tick_positions.append(x_pos)
                tick_labels.append(cond)
                x_pos += 1

            group_end = x_pos - 1
            group_spans.append((group_start, group_end, PHASE_LABELS.get(phase, phase)))
            x_pos += PHASE_GAP

        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, fontsize=9, rotation=35, ha='right')
        for start, end, label in group_spans:
            ax.text((start + end) / 2, ax.get_ylim()[1], label,
                    ha='center', va='bottom', fontsize=12,
                    fontweight='bold', color='#444444')
        ax.set_ylabel(ylabel, fontsize=12)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.legend(handles=legend_handles, fontsize=8, loc='upper right')
        fig.suptitle(f'{title} — {ylabel}', fontsize=14)
        fig.tight_layout(rect=[0, 0, 1, 0.93])
        figures.append(fig)

    return figures


DEFAULT_STRIP_STATS = [
    ('n_trials', 'N Trials'),
    ('accuracy', 'Accuracy'),
    ('win_stay', 'Win-Stay'),
    ('lose_shift', 'Lose-Shift'),
]

## Assemble & Save

In [ ]:
def generate_animal_report(animal, aid, strip_stats=DEFAULT_STRIP_STATS):
    genotype = animal.genotype
    tag = f'{aid} ({genotype.upper()})'
    opto = is_opto_cohort(animal)
    cohort = 'opto' if opto else 'non-opto'
    figures = []

    figures.append(plot_timeline(animal, aid))

    # ── Full data ─────────────────────────────────────────────
    uni_clean, uni_psyc, uni_um = compute_phase(animal, 'uniform', cohort)
    if opto:
        figures.append(plot_uniform_psychometric(uni_psyc, tag))
        figures.append(plot_uniform_um(uni_um, tag))
    else:
        fig = plot_phase_single(uni_psyc, uni_um, f'{tag} — Uniform')
        if fig is not None:
            figures.append(fig)

    ha_clean, ha_psyc, ha_um = compute_phase(animal, 'hard_a', cohort)
    hb_clean, hb_psyc, hb_um = compute_phase(animal, 'hard_b', cohort)
    has_hard = (any(v is not None for v in ha_psyc.values()) or
                any(v is not None for v in hb_psyc.values()))
    if has_hard:
        if opto:
            figures.append(plot_hard_psychometric(ha_psyc, hb_psyc, tag))
            figures.append(plot_hard_um(ha_um, hb_um, tag))
        else:
            fig = plot_nonopto_hard(ha_psyc, hb_psyc, ha_um, hb_um, tag)
            if fig is not None:
                figures.append(fig)

    # ── Downsampled (matched n) ───────────────────────────────
    if opto:
        ds_uni_psyc, ds_uni_um = downsample_phase(uni_clean, n_repeats=100)
        figures.append(plot_uniform_psychometric(ds_uni_psyc, f'{tag} — Uniform (matched n)'))
        figures.append(plot_uniform_um(ds_uni_um, f'{tag} — Uniform UMs (matched n)'))

        if has_hard:
            ds_ha_psyc, ds_ha_um = downsample_phase(ha_clean, n_repeats=100)
            ds_hb_psyc, ds_hb_um = downsample_phase(hb_clean, n_repeats=100)
            figures.append(plot_hard_psychometric(
                ds_ha_psyc, ds_hb_psyc, f'{tag} — Hard (matched n)'))
            figures.append(plot_hard_um(
                ds_ha_um, ds_hb_um, f'{tag} — Hard UMs (matched n)'))

    # ── Strip ─────────────────────────────────────────────────
    stat_cols = [col for col, _ in strip_stats]
    strip_df = build_strip_df(animal, stats_of_interest=stat_cols, cohort=cohort)
    figures.extend(plot_strip(strip_df, strip_stats, tag))

    return figures, strip_df


def save_pdf(figures, path):
    with PdfPages(path) as pdf:
        for fig in figures:
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

In [ ]:
def plot_phase_single(psyc, um, title):
    keys = [k for k in psyc if psyc[k] is not None and psyc[k].get('success')]
    if not keys:
        return None
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    key = keys[0]
    plot_psychometric(psyc[key], ax=axes[0], color=CC.get(key, 'grey'),
                      label=f'{key} (n={psyc[key]["n_trials"]})')
    axes[0].legend(fontsize=8); axes[0].set_title('Psychometric')
    u = um.get(key)
    if u is not None:
        plot_um(u, ax=axes[1], title=f'{key} (n={u["n_trials"]})',
                vmin=-0.35, vmax=0.35, colorbar=False)
    fig.suptitle(title, fontsize=14); fig.tight_layout()
    return fig


def plot_nonopto_hard(psyc_a, psyc_b, um_a, um_b, tag):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    pa, pb = psyc_a.get('regular'), psyc_b.get('regular')
    if pa and pa.get('success'):
        plot_psychometric(pa, ax=axes[0], color=HARD_A_COLOUR, label=f'Hard-A (n={pa["n_trials"]})')
    if pb and pb.get('success'):
        plot_psychometric(pb, ax=axes[0], color=HARD_B_COLOUR, label=f'Hard-B (n={pb["n_trials"]})')
    axes[0].set_title('A vs B: Psychometric', fontsize=11); axes[0].legend(fontsize=8)
    for i, (um_d, dist) in enumerate([(um_a, 'Hard-A'), (um_b, 'Hard-B')]):
        u = um_d.get('regular')
        if u is not None:
            plot_um(u, ax=axes[i+1], title=f'{dist} (n={u["n_trials"]})',
                    vmin=-0.35, vmax=0.35, colorbar=False)
        else:
            axes[i+1].text(0.5, 0.5, f'{dist}\nNo data', ha='center',
                           va='center', transform=axes[i+1].transAxes, color='grey')
    fig.suptitle(f'{tag} — Hard-A vs Hard-B', fontsize=14); fig.tight_layout()
    return fig

## Interactive: Single Animal

In [2]:
animal_id = 'SS14'  # ← change this
animal = experiment.get_animal(animal_id)
tag = f'{animal_id} ({animal.genotype.upper()})'
opto = is_opto_cohort(animal)
print(f'{animal_id} | {animal.genotype} | {"opto" if opto else "non-opto"}')

NameError: name 'is_opto_cohort' is not defined

{'um': array([[-0.01348966, -0.03594072, -0.03644021, -0.00848874,  0.05304854,
          0.01558646,  0.01436666,  0.00578657],
        [-0.03143012, -0.05347424, -0.05312784, -0.02642957,  0.11760309,
          0.05154958,  0.01199276,  0.04367008],
        [-0.08329137, -0.09617563, -0.09488024, -0.07852566,  0.17858171,
          0.18286611,  0.03276948,  0.09425308],
        [-0.17507892, -0.12969846, -0.15497921, -0.18898107,  0.20313612,
          0.35694158,  0.10949732,  0.13566106],
        [-0.14009818, -0.07359104, -0.18116121, -0.36001207,  0.17252578,
          0.38756324,  0.20275407,  0.14524442],
        [ 0.01732316, -0.02275826, -0.12808853,  0.05083545,  0.10631949,
          0.25932107,  0.21458143,  0.12455331],
        [-0.04694907, -0.0867961 , -0.03952192, -0.10895047,  0.04693962,
          0.10910072,  0.14243536,  0.09761489],
        [-0.13950702, -0.16964475,  0.01221039, -0.20535695,  0.01956025,
          0.01332716,  0.06656847,  0.08336518]]),
 'condit

In [ ]:
_, uni_psyc, uni_um = compute_phase(animal, 'uniform')
if opto:
    fig = plot_uniform_psychometric(uni_psyc, tag)
    plt.show()
    fig = plot_uniform_um(uni_um, tag)
    plt.show()

In [ ]:
_, ha_psyc, ha_um = compute_phase(animal, 'hard_a')
_, hb_psyc, hb_um = compute_phase(animal, 'hard_b')
if opto:
    fig = plot_hard_psychometric(ha_psyc, hb_psyc, tag)
    plt.show()
    fig = plot_hard_um(ha_um, hb_um, tag)
    plt.show()

### Interactive: Downsampled

In [ ]:
# Debug: check what downsample_phase actually does
uni_clean, uni_psyc, uni_um = compute_phase(animal, 'uniform')

# Check trial counts per condition
print("Trial counts per condition:")
for label, sessions in uni_clean.items():
    if sessions is None:
        print(f"  {label}: None")
        continue
    n = sum(s.trials.valid_mask.sum() for s in sessions)
    print(f"  {label}: {n} trials across {len(sessions)} sessions")

# Pool and check
from behav_utils.analysis.psychometry import _fit_psychometric_once
import warnings

stim_pools = {}
for label, sessions in uni_clean.items():
    if sessions is None:
        continue
    stim, choice = [], []
    for sess in sessions:
        mask = sess.trials.valid_mask
        stim.append(sess.trials.stimulus[mask])
        choice.append(sess.trials.choice[mask])
    stim_pools[label] = (np.concatenate(stim), np.concatenate(choice))
    print(f"  {label} pooled: stim {stim_pools[label][0].shape}, "
          f"choice unique={np.unique(stim_pools[label][1])}")

min_n = min(len(s) for s, c in stim_pools.values())
print(f"\nmin_n = {min_n}")

In [ ]:
# Try a single fit on subsampled data
rng = np.random.default_rng(42)
x_fit = np.linspace(-1, 1, 100)

test_label = list(stim_pools.keys())[0]
s_all, c_all = stim_pools[test_label]
idx = rng.choice(len(s_all), min_n, replace=True)
s_sub, c_sub = s_all[idx], c_all[idx]

print(f"Subsample from '{test_label}': {len(s_sub)} trials")
print(f"  stim range: [{s_sub.min():.3f}, {s_sub.max():.3f}]")
print(f"  choice values: {np.unique(c_sub)}")
print(f"  choice mean: {c_sub.mean():.3f}")

try:
    fit = _fit_psychometric_once(s_sub, c_sub, x_fit)
    print(f"  fit success: {fit.get('success')}")
    print(f"  fit keys: {list(fit.keys())}")
    if fit.get('success'):
        print(f"  mu={fit['mu']:.3f}, sigma={fit['sigma']:.3f}")
        if 'y_fit' in fit:
            print(f"  y_fit shape: {np.array(fit['y_fit']).shape}")
        else:
            print("  WARNING: no 'y_fit' key in fit result!")
except Exception as e:
    print(f"  FIT FAILED: {e}")

In [ ]:
# Full downsample
ds_uni_psyc, ds_uni_um = downsample_phase(uni_clean, n_repeats=50)

print("Downsampled results:")
for label in uni_clean:
    p = ds_uni_psyc.get(label)
    u = ds_uni_um.get(label)
    if p is None:
        print(f"  {label}: psyc=None")
    else:
        print(f"  {label}: psyc n={p['n_trials']}, success={p['success']}, "
              f"n_fits={p.get('n_fits', '?')}")
    if u is None:
        print(f"  {label}: um=None")
    else:
        print(f"  {label}: um n={u['n_trials']}")

In [ ]:
# Plot downsampled if available
if any(v is not None for v in ds_uni_psyc.values()):
    fig = plot_uniform_psychometric(ds_uni_psyc, f'{tag} — Uniform (matched n)')
    plt.show()
else:
    print("All downsampled psychometrics are None — check diagnostics above")

if any(v is not None for v in ds_uni_um.values()):
    fig = plot_uniform_um(ds_uni_um, f'{tag} — Uniform UMs (matched n)')
    plt.show()

In [ ]:
# Strip — one page per stat
strip_df = build_strip_df(animal)
for fig in plot_strip(strip_df, DEFAULT_STRIP_STATS, tag):
    plt.show()

In [ ]:
# Custom stats
custom_stats = [('accuracy', 'Accuracy'), ('win_stay', 'Win-Stay'),
                ('lose_shift', 'Lose-Shift'), ('side_bias', 'Side Bias')]
strip_df2 = build_strip_df(animal, stats_of_interest=[c for c, _ in custom_stats])
for fig in plot_strip(strip_df2, custom_stats, tag):
    plt.show()

## Batch: All Animals → PDF

In [ ]:
OUTPUT_DIR = Path('outputs/animal_reports_2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for aid, animal in sorted(experiment.animals.items()):
    cohort = 'opto' if is_opto_cohort(animal) else 'non-opto'
    print(f'{aid} | {animal.genotype} | {cohort} ...', end=' ')
    figures, strip_df = generate_animal_report(animal, aid)
    pdf_path = OUTPUT_DIR / f'{aid}_{animal.genotype}_{cohort}.pdf'
    save_pdf(figures, pdf_path)
    print(f'{len(figures)} pages → {pdf_path.name}')

print(f'\nDone. Reports in {OUTPUT_DIR}/')